In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
#For the drift measurements we will use the evidently library
from evidently import Report
from evidently.metrics import *
from evidently.presets import *

In [3]:
BASELINE_PATH = "../data/final/power_tetouan_city_after_EDA.csv"      
ALTERED_PATH = "../data/modified/power_tetouan_city_altered.csv" 

def simulate_shift():

    df = pd.read_csv(BASELINE_PATH)

    # Example drift simulations:
    df_drift = df.copy()

    # 1. Desplazamiento de media
    numeric_cols = df_drift.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df_drift[col] = df_drift[col] + np.random.uniform(1.0, 5.0)

    # 2. Eliminando una columna al azar
    col_to_drop = np.random.choice(df_drift.columns)
    df_drift = df_drift.drop(columns=[col_to_drop])

    df_drift.to_csv(ALTERED_PATH, index=False)
    print(f"Simulated drift data saved to {ALTERED_PATH}")

def load_baseline():
    print(f"Loading original data from {BASELINE_PATH}")
    return pd.read_csv(BASELINE_PATH)

def load_altered():
    print(f"Loading altered data from {ALTERED_PATH}")
    return pd.read_csv(ALTERED_PATH)

In [4]:
def generate_drift_report(df_baseline,df_altered):
    '''
    Generates data drift report using the evidently library
    '''

    report = Report([
        DataSummaryPreset(),
        DataDriftPreset()
    ], include_tests=True)

    #timestamp for versioning
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    my_eval = report.run(current_data = df_altered,
                        reference_data=df_baseline)
    
    report_path = f'./reports/drift_report_{timestamp}.html'

    my_eval.save_html(report_path)

    print(f'Drift report generated at: {report_path}')

In [55]:
if __name__ == "__main__":
    simulate_shift()
    baseline = load_baseline()
    current = load_altered()
    generate_drift_report(baseline, current)

Simulated drift data saved to ../data/modified/power_tetouan_city_altered.csv
Loading original data from ../data/final/power_tetouan_city_after_EDA.csv
Loading altered data from ../data/modified/power_tetouan_city_altered.csv
Drift report generated at: ./reports/drift_report_20251115_154012.html


In [ ]:
#retraining
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42

In [17]:
data_path = BASELINE_PATH

df = pd.read_csv(data_path)
target = "z1_power_cons"

X = df.drop(columns=[target, "DateTime"], errors="ignore")
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

num_cols = selector(dtype_include=np.number)(X)
cat_cols = selector(dtype_exclude=np.number)(X)

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="warn"), cat_cols)
])

# Modelos base con Pipelines
rf_model = Pipeline([
    ("prep", prep),
    ("model", RandomForestRegressor(
        n_estimators=120, 
        max_depth=12, 
        random_state=RANDOM_SEED
    ))
])

rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

baseline = pd.DataFrame(
    [["RandomForest", mae, rmse, r2]],
    columns=["Model", "MAE", "RMSE", "R2"]
)

baseline

,Model,MAE,RMSE,R2
0,RandomForest,677.406192,988.655524,0.981254


In [18]:
data_path = ALTERED_PATH

df_drifted = pd.read_csv(ALTERED_PATH)

X_drift = df_drifted.drop(columns=[target, "DateTime"], errors="ignore")
y_drift = df_drifted[target]

for col in X_train.columns:
    if col not in X_drift.columns:
        X_drift[col] = np.nan

# Keep columns in same order as training
X_drift = X_drift[X_train.columns]

pred_drift = rf_model.predict(X_drift)

mae_drift = mean_absolute_error(y_drift, pred_drift)
rmse_drift = mean_squared_error(y_drift, pred_drift)**0.5
r2_drift = r2_score(y_drift, pred_drift)

altered = pd.DataFrame(
    [["RandomForest (drifted)", mae_drift, rmse_drift, r2_drift]],
    columns=["Model", "MAE", "RMSE", "R2"]
)
altered

,Model,MAE,RMSE,R2
0,RandomForest (drifted),2875.792332,3901.977126,0.705595
